# L26 — Warmup Detection and Welch's Method

**Module**: M08 | **Chapter**: 10 | **Lecture**: L26

## Learning Objectives
By the end of this notebook you will be able to:
1. Explain why an empty-and-idle initial condition biases steady-state estimates.
2. Construct a batch-mean trajectory from a single long simulation run.
3. Apply Welch's moving-average smoother to identify the warmup period.
4. Quantify the reduction in bias achieved by warmup deletion.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

We use an M/M/1 queue (λ=0.8, μ=1.0, ρ=0.8) throughout.
True steady-state: W_q* = λ/(μ(μ−λ)) = 4.0 minutes.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
from scipy import stats

## 1. Simulate a Long Run

In [ ]:
def run_mm1_long(lam, mu, n_customers, seed):
    """Single-server M/M/1, return per-customer wait in queue."""
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=1)
    waits = []

    def customer():
        arrival = env.now
        with server.request() as req:
            yield req
            waits.append(env.now - arrival)
            yield env.timeout(rng.exponential(1.0 / mu))

    def arrivals():
        for _ in range(n_customers):
            env.process(customer())
            yield env.timeout(rng.exponential(1.0 / lam))

    env.process(arrivals())
    env.run()
    return np.array(waits)


LAM, MU = 0.8, 1.0
WQ_STAR = LAM / (MU * (MU - LAM))   # = 4.0

print(f"M/M/1: λ={LAM}, μ={MU}, ρ={LAM/MU}, Wq* = {WQ_STAR:.2f} min")

waits = run_mm1_long(LAM, MU, n_customers=50_000, seed=42)
print(f"Simulated {len(waits)} customers. Overall mean Wq = {waits.mean():.3f}")

## 2. Batch Mean Trajectory

In [ ]:
BATCH_SIZE = 100
n_batches = len(waits) // BATCH_SIZE
batch_means = waits[:n_batches * BATCH_SIZE].reshape(n_batches, BATCH_SIZE).mean(axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.arange(1, n_batches+1), batch_means, color='steelblue', lw=0.8, alpha=0.7,
        label='Batch mean Wq')
ax.axhline(WQ_STAR, color='black', lw=1.5, linestyle='--', label=f'Wq* = {WQ_STAR:.1f}')
ax.axhline(WQ_STAR * 1.1, color='tomato', lw=1, linestyle=':', label='±10% band')
ax.axhline(WQ_STAR * 0.9, color='tomato', lw=1, linestyle=':')
ax.set_xlabel(f'Batch number (batch size = {BATCH_SIZE} customers)')
ax.set_ylabel('Batch mean Wq (min)')
ax.set_title(f'M/M/1 batch mean trajectory  (50k customers, λ={LAM}, μ={MU})')
ax.set_xlim(0, 200)   # zoom into early transient
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# First batch that enters the ±10% band
in_band = np.where(
    (batch_means > WQ_STAR * 0.9) & (batch_means < WQ_STAR * 1.1)
)[0]
print(f"First batch inside ±10% band: batch {in_band[0]+1} (customer {(in_band[0]+1)*BATCH_SIZE})")

## 3. Welch's Moving Average

Welch's method smooths the noisy batch-mean series to reveal the trend:
$$\bar{Y}_w(b) = \frac{1}{2w+1} \sum_{j=b-w}^{b+w} \bar{W}_q^{(j)}$$

A larger window $w$ produces a smoother curve but pushes the estimate further from the edges.

In [ ]:
def welch_smooth(batch_means, w):
    """Apply Welch's moving average with window w (edge-truncated)."""
    n = len(batch_means)
    smoothed = np.full(n, np.nan)
    for b in range(w, n - w):
        smoothed[b] = batch_means[b-w : b+w+1].mean()
    return smoothed


fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(batch_means[:300], color='steelblue', lw=0.6, alpha=0.4, label='Raw batch means')

colors = ['tab:orange', 'tab:green', 'tab:red']
for w, col in zip([5, 10, 20], colors):
    sm = welch_smooth(batch_means, w)
    ax.plot(sm[:300], color=col, lw=2, label=f'Welch w={w}')

ax.axhline(WQ_STAR,       color='black', lw=1.5, linestyle='--', label=f'Wq*={WQ_STAR:.1f}')
ax.axhline(WQ_STAR * 1.1, color='grey',  lw=1,   linestyle=':')
ax.axhline(WQ_STAR * 0.9, color='grey',  lw=1,   linestyle=':')
ax.set_xlabel(f'Batch number (batch size = {BATCH_SIZE})')
ax.set_ylabel('Smoothed batch mean Wq (min)')
ax.set_title("Welch's method — identifying warmup period")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Choose warmup based on w=10
sm10 = welch_smooth(batch_means, 10)
stable = np.where(
    (sm10 > WQ_STAR * 0.9) & (sm10 < WQ_STAR * 1.1)
)[0]
warmup_batches = stable[0] if len(stable) > 0 else 0
warmup_customers = warmup_batches * BATCH_SIZE
print(f"Welch w=10: warmup ≈ {warmup_batches} batches = {warmup_customers} customers")

## 4. Bias Reduction from Warmup Deletion

We compare estimates with and without warmup deletion across 30 replications.

In [ ]:
N_REPS       = 30
N_CUSTOMERS  = 5_000
WARMUP_CUST  = warmup_customers

wq_all    = []   # include all customers
wq_trimmed = []  # delete warmup

for seed in range(N_REPS):
    w = run_mm1_long(LAM, MU, n_customers=N_CUSTOMERS, seed=seed)
    wq_all.append(w.mean())
    wq_trimmed.append(w[WARMUP_CUST:].mean() if WARMUP_CUST < len(w) else w.mean())

wq_all    = np.array(wq_all)
wq_trimmed = np.array(wq_trimmed)

def ci_95(x):
    m = x.mean()
    h = stats.t.ppf(0.975, df=len(x)-1) * x.std(ddof=1) / np.sqrt(len(x))
    return m, h

m_all, h_all       = ci_95(wq_all)
m_trim, h_trim     = ci_95(wq_trimmed)

print(f"Including all customers:  Wq = {m_all:.3f} ± {h_all:.3f}  (95% CI: [{m_all-h_all:.3f}, {m_all+h_all:.3f}])")
print(f"After warmup deletion:    Wq = {m_trim:.3f} ± {h_trim:.3f}  (95% CI: [{m_trim-h_trim:.3f}, {m_trim+h_trim:.3f}])")
print(f"True Wq* = {WQ_STAR:.3f}")
print()
print(f"CI without warmup deletion contains Wq*? {m_all - h_all <= WQ_STAR <= m_all + h_all}")
print(f"CI with    warmup deletion contains Wq*? {m_trim - h_trim <= WQ_STAR <= m_trim + h_trim}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([wq_all, wq_trimmed], labels=['All customers', f'Warmup={WARMUP_CUST} deleted'])
ax.axhline(WQ_STAR, color='red', linestyle='--', lw=1.5, label=f'Wq* = {WQ_STAR}')
ax.set_ylabel('Estimated Wq (min)')
ax.set_title(f'Effect of warmup deletion  (n={N_REPS} reps, {N_CUSTOMERS} customers each)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Effect of Load on Warmup Length

Higher utilisation means longer transients — the system takes longer to fill up.

In [ ]:
rho_values = [0.5, 0.7, 0.8, 0.9]

fig, axes = plt.subplots(1, len(rho_values), figsize=(14, 3), sharey=False)

for ax, rho in zip(axes, rho_values):
    lam_r = rho * MU
    wq_star_r = lam_r / (MU * (MU - lam_r))
    w_r = run_mm1_long(lam_r, MU, n_customers=20_000, seed=42)
    b_means = w_r[:20_000 // BATCH_SIZE * BATCH_SIZE].reshape(-1, BATCH_SIZE).mean(axis=1)
    ax.plot(b_means[:100], color='steelblue', lw=0.8, alpha=0.7)
    ax.axhline(wq_star_r, color='black', lw=1.5, linestyle='--')
    ax.set_title(f'ρ={rho}  Wq*={wq_star_r:.1f}')
    ax.set_xlabel('Batch')
    if ax == axes[0]: ax.set_ylabel('Batch mean Wq')
    ax.grid(True, alpha=0.3)

plt.suptitle('Warmup length increases with utilisation', fontsize=11)
plt.tight_layout()
plt.show()

---
## Try It Yourself

1. **Warm start**: Instead of starting empty, initialise the queue with 4 customers already present (matching the steady-state mean). Does the transient bias disappear? How does the batch-mean trajectory look compared to the empty-start case?

2. **Batch size sensitivity**: Repeat Welch's method with batch sizes of 25, 50, 100, and 200 customers. How does the choice of batch size affect the apparent warmup length and the smoothness of the Welch curve?

3. **Terminating simulation**: A bank opens at 09:00 and closes at 17:00. Is Welch's warmup analysis appropriate here? What would you do instead? Write 3–4 sentences explaining the terminating vs. steady-state distinction.